In [46]:
import sys

In [47]:
%%capture
try:
    # Attempt to import a module that's only available in Colab
    from google.colab import drive

    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    # Colab specific setup
    drive.mount("/content/drive")
    sys.path.append("/content/drive/MyDrive/structure-loss-classification/")
    my_local_data = "/content/drive/MyDrive/types/"
    %cd '/content/drive/MyDrive/structure-loss-classification/'
    %pip install -r requirements.txt
else:
    # Local machine setup
    my_local_data = "/mnt/g/My Drive/types/"

In [48]:
from sklearn.metrics import ConfusionMatrixDisplay
import torchvision.transforms as transforms
import torch
from sklearn.model_selection import train_test_split

In [49]:
import pickle

In [50]:
from models.models import LeNet5, ResNet18
from lightning_modules.lightning_modules import LitLeNet5, LitResNet18, LitVGG16
from visualization.filters import display_filters
from visualization.display import process_plot_image, display_metrics
from datasets.data_modules import CustomImageDataModule, CustomImageDataModule2
from train.train import get_features, train_model, train_with_cv
from hyperparameter_tuning.tune import HyperParameterTuner
from datasets.datasets import CustomDatasetWrapper
from utils.utils import load_targets, get_stat_metrics, get_train_val_data

In [51]:
from ray import tune

In [52]:
toTensorAndNormalize = transforms.Compose(
    [
        transforms.Resize((244, 244)),
        transforms.RandomHorizontalFlip(),
        # transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # mean  # std
    ]
)

In [53]:
classification_mode = "only_bad"

In [54]:
task = {"binary": 2, "only_bad": 3, "all": 4}

In [55]:
num_classes = task[classification_mode]

In [56]:
aux_data = CustomDatasetWrapper(
    root_dir=my_local_data,
    classification_mode=classification_mode,
    transform=toTensorAndNormalize,
)

In [57]:
targets = load_targets(aux_data)

In [58]:
aux_data.label_map

{1: 0, 2: 1, 3: 2}

In [59]:
datamodule = CustomImageDataModule2(dataset=aux_data, targets=targets)

In [60]:
model_class = LitLeNet5

In [61]:
trainer_config = {
    "patience": 10,
    "accelerator": "gpu",
    "devices": -1,
    "max_epochs": 200,
    "precision": 32,
    "n_steps": 50,
}

torch.set_float32_matmul_precision('high')

save_dir = f'logdir/pipeline_1/{model_class.__name__}/{classification_mode}/cv'

In [62]:
# search_space = {
#    'model_params': {"num_classes": num_classes,
#                     "size_layer_1": tune.choice([100, 84, 50]),
#                     "size_layer_2": tune.choice([30, 20, 10]),
#                     "learning_rate": tune.loguniform(1e-4, 1e-2)},
   
#     "batch_size": tune.choice([32, 64, 128]),
# }

search_space = {
   'model_params': {"num_classes": num_classes,
                    "size_layer_1": tune.choice([2, 4, 6]),
                    "size_layer_2": tune.choice([1, 3, 5]),
                    "learning_rate": tune.loguniform(1e-4, 1e-2)},
   
    "batch_size": tune.choice([2, 4, 8]),
}

default_config = {
    'num_classes': num_classes,
    'learning_rate': 0.001
}

In [63]:
# Try to load cached targets first
targets = load_targets(aux_data)

In [64]:
my_tuner = HyperParameterTuner(model_class = model_class,
                                   datamodule = datamodule,
                                   search_space = search_space,
                                   num_samples = 8,
                                   num_epochs = 10)

In [65]:
torch.cuda.device_count()

0

In [66]:
my_tuner.resources

{'num_cpus': 8, 'num_gpus': 0}

In [ ]:
params = my_tuner.hypertune()

### Test train

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
from train.train import incremental_training

In [ ]:
model_class = LitVGG16

In [ ]:
model_params = {
    "num_classes": num_classes,
    "size_layer_1": 128,
    "size_layer_2": 64,
    "size_layer_3": 32,
    "learning_rate": 8.479282111072422e-05,
}

In [ ]:
trainer_config = {
    "patience": 50,
    "accelerator": "gpu",
    "devices": -1,
    # "max_epochs": 100,
    "precision": 32,
    "n_steps": 5,
    "save_dir": f"logdir/LitVGG16/{classification_mode}",
}

In [ ]:
# metrics = incremental_training(
#     model_class=model_class,
#     model_params=model_params,
#     trainer_config=trainer_config,
#     data_module=data_module,
#     early_stop_patience=5,

# )

In [ ]:
from torchvision.models.feature_extraction import get_graph_node_names

In [ ]:
from models.models import ResNet18

In [ ]:
model = LeNet5(
    num_classes=num_classes,
)
model = model.to("cuda")

In [ ]:
# get_graph_node_names(model)[0]

In [ ]:
# layers = [e for e in get_graph_node_names(model)[0] if 'pool' in e]
layers = [
    "convStack.1",
    "convStack.2",
    "convStack.3",
]

In [ ]:
features, labels = get_features(
    model=model,
    layers=layers,
    data_loader=aux_data,
    device="cuda",
    classification_mode=classification_mode,
)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from visualization.filters import display_filters

In [ ]:
display_filters(
    features=features,
    img_num=33,
    layers=layers,
    cmap="viridis",
    name_layers=["pool 1", "pool 2", "pool 3"],
    data=aux_data,
)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay

In [ ]:
import numpy as np

In [ ]:
new_features = {}

In [ ]:
for layer_name in features:
    new_features[layer_name] = [feature.flatten() for feature in features[layer_name]]

In [ ]:
hp_sweep = {
    "kernel": "rbf",
    "C": 3.2010661148711947,
    "gamma": "scale",
}

In [ ]:
from hyperparameter_tuning.tune import hypertune_classifier, SKLearnHyperParameterTuner

In [ ]:
param_grid = {
    'kernel': tune.choice(['linear', 'poly', 'rbf', 'sigmoid']),
    'C': tune.loguniform(1e-3, 1e3),
    'gamma': tune.choice(['scale', 'auto']),
}

In [ ]:
X = np.array(new_features[layers[1]])
y = np.array(labels)

In [ ]:
my_tuner = SKLearnHyperParameterTuner(model = SVC,
                                      search_space = param_grid,
                                      X= X,
                                      y = y,
                                      use_gpu=False)

In [ ]:
best_params = my_tuner.hypertune()

In [ ]:
best_params

In [ ]:
import ray

In [ ]:
ray.shutdown()

In [ ]:
ray.init(resources={'num_cpus':12, 'num_gpus':1})

In [ ]:
torch.cuda.is_available()

In [ ]:
print(ray.cluster_resources())

In [ ]:
# Use the first layer's features in this example
X = np.array(features[layers[hp_sweep["features_index"]]])
y = np.array(labels)

# Initialize the 5-fold cross-validator
cv = StratifiedKFold(n_splits=5)

# Create an SVC with an RBF kernel
# svc = SVC(kernel=hp_sweep['kernel'],
#           C=hp_sweep['C'],
#           gamma=hp_sweep['gamma'])

svc = SVC()

# Initialize an empty confusion matrix to store the summary of all folds
summary_confusion_matrix = np.zeros((len(np.unique(y)), len(np.unique(y))))
accuracies = []

In [ ]:
aux_data.classes

In [ ]:
names_dict = {
    "typeA": "Diameter\nFluctuations",
    "typeB": "Node Cut",
    "typeC": "Particle Hit",
    "goodIngots": "No Structure\nLoss",
}

# Get class indices from the ImageFolder
class_indices = aux_data.classes

# Order your names list according to the class indices
names = [names_dict[class_name] for class_name in aux_data.classes]

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
dfs = []

In [ ]:
for folder in sorted(glob.glob("logdir/all/lightning_logs/*")):
    df = pd.read_csv(f"{folder}/metrics.csv")
    dfs.append(df)

In [ ]:
dfs

In [ ]:
for c, df in enumerate(dfs):
    # Plot the main line with markers
    plt.plot(df.epoch, df.val_accuracy, '-', label=str(c+1))

    # Find the index of the maximum val_accuracy in the current dataframe
    max_val_acc_index = df.val_accuracy.idxmax()

    # If this is not the last dataframe, draw a dotted line to the first point of the next dataframe
    if c < len(dfs) - 1:
        next_df = dfs[c + 1]
        # Get the point with max val_accuracy from the current dataframe and the first point of the next dataframe
        x_values = [df.epoch.iloc[max_val_acc_index], next_df.epoch.iloc[0]]
        y_values = [df.val_accuracy.iloc[max_val_acc_index], next_df.val_accuracy.iloc[0]]
        # Plot with dotted lines
        plt.plot(x_values, y_values, 'o-', color='grey')

# plt.legend()
plt.show()


In [ ]:
dir_pre = "/mnt/g/My Drive/structure-loss-classification/results/LitResNet18/"
dir_not = (
    "/mnt/g/My Drive/structure-loss-classification/results/LitResNet18-not-pretrained"
)

In [ ]:
from visualization.display import compare_resnet18

In [ ]:
compare_resnet18("only_bad")

In [ ]:
compare_resnet18("binary")

In [ ]:
compare_resnet18("all")